### EEG Preprocessing Pipeline (draft)

This notebook contains my preliminary EEG preprocessing sequence. <br> <br>The steps that are included and their order is mainly informed by the **COBIDAS report** on "Best Practices in Data Analysis and Sharing in Neuroimaging using MEEG" (Pernet et al., 2018). I will adjust the sequence of steps based on some practical considerations, e.g., I will use the **Autoreject function** as a tool for automated artifact rejection and correction (Jas et al., 2017).

**Planned sequence** <br>
1. BIDS conversion
1. Visual inspection of raw continuous data 
2. Bad channel identification and interpolation
3. Filtering: high-pass, notch, low-pass 
4. Downsampling
5. Preliminary epoching ("data segmentation") + light artefact screen to exclude very bad segments before ICA
6. ICA 
7. Identifying and removing non-brain ICA components
8. Final epoching (Splitting into Encoding and Retrieval), then MERGE with behavioral data
9. Autoreject on epochs
10. Baseline correction
11. Re-referencing ([] decide if necessary)
12. ERP averaging and permutation testing
( 13. Multivariate analyses ) - They do not need this until September!


Later, decide on final parameters for pre-processing, e.g,.
[] which event to look at for topoplots / channel inspection, either fixation or maybe CUE is better!!
[] filters etc



Also later: Lerntagebuch für EEG konzepte, now we focus on the code because I have deliverables

Next week (10.7.-11.7.)

1) Notebook structure
Five is a bit much, but the answer isn't just "merge everything" — it's splitting along a different axis: what needs manual/visual input per subject vs. what's fully automatable. That distinction matters a lot once you get to point 4 (40 participants).
Suggested structure:

Notebook 1 — BIDS conversion: stays separate. Different input/output format, typically run once per subject with minimal manual intervention, easy to batch later.
Notebook 2 — Interactive QC: visual inspection of raw + bad channel identification. This one has to stay somewhat manual/per-subject (you're looking at plots and making judgment calls), so don't try to fully automate it away. Output: a bad-channel list saved per subject (e.g., JSON or CSV), not raw data itself.
Notebook 3 — Automated preprocessing: filtering → downsampling → preliminary epoching → ICA fit → component identification/removal → final epoching → merge behavioral metadata → Autoreject → baseline → re-reference. This is your current notebooks 3+4+part of the epoching, merged. Since each step is fast and mostly deterministic/automatable (EOG-based ICA detection, Autoreject), this is the one you'll eventually want to turn into a loop over subjects rather than a one-off notebook.
Notebook 4 — Averaging + permutation testing: stays separate, group-level, conceptually distinct analysis stage.

So: 4 notebooks, with Notebook 3 doing more work than before but staying coherent because it's "the automated middle chain" as one conceptual unit.
2) Understanding the code + documentation
This is worth doing while you restructure, not after — you'll understand things better by rewriting with reasons attached than by re-reading old code. A structure that's worked well for others in your position:

For every step, add a markdown cell with: (a) what the step does mechanically, (b) why it's in this position in the sequence (dependencies on prior steps), (c) which parameter choices you made and why (cite COBIDAS/Autoreject papers where relevant), (d) what you'd say if asked about it in your defense.
Since you're already following COBIDAS, you basically have a built-in citation trail for every step — lean on that.
I'd genuinely enjoy going through your parameter choices with you (filter cutoffs, epoch windows, ICA component count logic, Autoreject thresholds) if you want to do that as its own session — that's a good use of a dedicated notebook-cleanup pass rather than something to rush.

Scaling to 40 participants
This is the point that most affects your notebook structure. Notebook 3 (automated preprocessing) is the candidate for looping — but I'd suggest an intermediate step before you jump straight to "loop over 40 subjects":

Refactor the Notebook 3 logic into functions in a .py module (e.g., preprocessing_utils.py) parametrized by subject ID and paths. This gives you: reusability, easier debugging (test one function in isolation), and a notebook that becomes a thin orchestration layer rather than 40x copy-pasted cells.
Keep a single-subject "development" notebook (your current Notebook 3, now calling the module functions) for exploration/debugging on one subject, and a separate batch script or notebook that loops the same functions across all 40 subjects and logs outcomes (e.g., how many epochs survived Autoreject per subject — ties back to the trial-count check we discussed).
Notebook 2 (interactive QC) stays inherently per-subject and manual — that one doesn't loop, it's just something you'll run 40 times individually (or in batches with breaks).

[] For now, I can only do the pre-processing for participants 16 and onward. Before that,
the CSV files and the number of extracted epochs will not match... Need to discuss this with Javier and Laura.